# 04 · Heston known-model check

Owner: simulation lead.

Simulate a Heston world where the true minimum-variance correction is known, train the residual network on it, and compare the learned correction to the truth.

In [ ]:
import sys, pandas as pd, numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, "..")
from config import *
from src import load, pairs as pairs_mod, models, evaluate
from src import heston

In [ ]:
P = heston.HestonParams()
sim = heston.simulate_chain(P, n_days=1000, seed=0)   # a few minutes
sim_pairs = pairs_mod.assign_splits(pairs_mod.build_pairs(sim),
                                    splits={"train": (2016, 2018), "val": (2019, 2019), "test": (2020, 2020)})
sim_pairs["true_corr"] = heston.true_mv_correction(sim_pairs, P)
sim_pairs.groupby("split").size()

In [ ]:
feats = ["T", "delta", "moneyness", "iv", "vega", "gamma"]   # theta not simulated
tr, va = sim_pairs[sim_pairs.split == "train"], sim_pairs[sim_pairs.split == "val"]
nn = models.NeuralHedger(residual=True, features=feats).fit(tr, val=va)
sim_pairs["delta_nn_resid"] = nn.predict(sim_pairs)
sim_pairs["delta_true"] = sim_pairs["delta"] + sim_pairs["true_corr"]
evaluate.gain_table(sim_pairs[sim_pairs.split == "test"], ["nn_resid", "true"], ["type"])

In [ ]:
te = sim_pairs[sim_pairs.split == "test"]
plt.scatter(te["true_corr"], te["delta_nn_resid"] - te["delta"], s=2, alpha=.3)
lim = np.percentile(te["true_corr"], [1, 99]); plt.plot(lim, lim, "r")
plt.xlabel("true MV correction (Heston)"); plt.ylabel("learned correction"); plt.savefig(FIGURES_DIR / "heston_learned_vs_true.png", dpi=150)